In [ ]:
### SETUP 
import os
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from matplotlib import pyplot as plt
from torchsummary import summary
import helper
from tqdm import tqdm

ROOT = "/home/andre/courses/AdvMl_V26/A00_AstronomyCNN"
DATA = os.path.join(ROOT,"DATA")
LABEL_PATH = os.path.join(DATA,"labels.npy")
SPECTRA_PATH = os.path.join(DATA,"spectra.npy")

device = torch.device("cpu")


In [ ]:
# Download data
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="simbaswe/galah4", filename="labels.npy", repo_type="dataset", local_dir=DATA)
hf_hub_download(repo_id="simbaswe/galah4", filename="spectra.npy", repo_type="dataset", local_dir=DATA)

In [ ]:
spectra = np.load(SPECTRA_PATH)
spectra_length = spectra.shape[1]
# labels: mass, age, l_bol, dist, t_eff, log_g, fe_h, SNR
labelNames = ["mass", "age", "l_bol", "dist", "t_eff", "log_g", "fe_h", "SNR"]
labels = np.load(LABEL_PATH)
# We only use the three labels: t_eff, log_g, fe_h
labelNames = labelNames[-4:-1]
labels = labels[:, -4:-1]
n_labels = labels.shape[1]
spectra = np.log(np.maximum(spectra, 0.2))

In [ ]:
num_spectras = spectra.shape[0]

idx = np.random.choice(num_spectras, size=4, replace=False)
fig, axes = plt.subplots(4, 1, figsize=(6, 6), sharex=True)


for ax, i in zip(axes, idx):
    ax.plot(spectra[i])
    ax.set_ylabel(f"spectra {i}")

axes[-1].set_xlabel("Index")

plt.tight_layout()
plt.show()

In [ ]:
train_size = int(0.7*num_spectras)
val_size = int(0.15*num_spectras)
test_size = num_spectras-train_size-val_size

y_mean, y_std = labels.mean(axis=0), labels.std(axis=0)
y_data = (labels - y_mean)/y_std


x_tensor = torch.tensor(spectra,dtype=torch.float32).view(num_spectras,-1).to(device)
#x_tensor = nn.MaxPool1d(2)(x_tensor)
y_tensor = torch.tensor(y_data,dtype=torch.float32).view(num_spectras,-1).to(device)
train_dataset, val_dataset, test_dataset = random_split(TensorDataset(x_tensor,y_tensor),[train_size,val_size,test_size])

batch_size = 64
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=0,pin_memory=False)
val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=False)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=False)


In [ ]:
Nc = 3

class simpleCNN(nn.Module):
    def __init__(self):
        super(simpleCNN,self).__init__()
        stride=1
        kernel_size = 6         
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=1,out_channels=2**0*Nc,kernel_size=kernel_size,stride=stride),
            nn.ReLU(),
            nn.MaxPool1d(3),

            nn.Conv1d(in_channels=2**0*Nc,out_channels=2**1*Nc,kernel_size=kernel_size,stride=stride),
            nn.ReLU(),
            nn.MaxPool1d(3),

            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.LazyLinear(3),
        )
        
    def forward(self,x):
        return self.net(x)


In [ ]:
num_epoch = 30
# Model initialization
model = simpleCNN()
loss_func = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epoch, eta_min=1e-6)

In [ ]:


# Training
train_losses, val_losses = [], []

for epoch in range(num_epoch):
    print(f"Epoch {epoch+1}, LR: {scheduler.get_last_lr()[0]:.6f}")
    model.train()
    train_loss = 0
    for batch_x, batch_y in tqdm(train_loader):
        batch_x = batch_x.unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        predictions = model(batch_x)
        loss = loss_func(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    print(f"Loss {train_loss:.3f} Epoch {epoch+1}")

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.unsqueeze(1)
            predictions = model(batch_x)
            loss = loss_func(predictions, batch_y)
            val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
    print(f"Val Loss {val_loss:.3f} Epoch {epoch+1}")

    # Step the scheduler after each epoch
    scheduler.step()

In [ ]:
all_pred = []
all_true = []
# Validation data (assuming val_loader; change to test_loader if needed)
model.eval()
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.unsqueeze(1)
        predictions = model(batch_x)
        all_pred.append(predictions.cpu())  # move to CPU to save GPU memory
        all_true.append(batch_y.cpu())

# Concatenate all batches into single tensors, then convert to numpy
all_pred = torch.cat(all_pred, dim=0).numpy()
all_true = torch.cat(all_true, dim=0).numpy()

# Rescale back to original units using training mean/std
all_pred_original = all_pred * y_std + y_mean
all_true_original = all_true * y_std + y_mean

In [ ]:
label_names = ["T_eff", "LogG","Fe"]

fig, axs = plt.subplots(1,3,figsize=(12,4))
alpha = 0.5

for k,ax in enumerate(axs):

    ax.scatter(all_pred_original[:,k],all_true_original[:,k],alpha=alpha,s=1)
    ax.set_xlabel(f"Predicted {label_names[k]}")
    ax.set_ylabel(f"True {label_names[k]}")

plt.tight_layout()
plt.savefig("predictions.pdf",dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(5,3))
plt.plot(train_losses,label="Train")
plt.plot(val_losses,label="Val")
plt.ylabel("MSE Loss (log-scale)")
plt.xlabel("Epoch")
plt.yscale("log")
plt.legend()
plt.tight_layout()
plt.savefig("loss.pdf",dpi=300)
plt.show()